# HOLT-WINTERS SEASONAL MODELING
**Sector:** Sports  |  **Table:** `court_bookings`
**By:** SURNAME, FIRSTNAME  &  SURNAME, FIRSTNAME

## 1. Load and explore data

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# TODO: replace with YOUR raw GitHub link to court_bookings.csv
URL = "https://raw.githubusercontent.com/<username>/<repo>/main/court_bookings.csv"

df = pd.read_csv(URL, parse_dates=["date"], index_col="date")
df.index.freq = "D"          # tell pandas the data is daily
series = df["court_bookings"]

In [ ]:
# Preview the first five rows
df.head()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Line plot of the time series
plt.figure(figsize=(12, 4))
plt.plot(series, color="steelblue")
plt.title("Daily Court Bookings - HoopZone Sports Complex")
plt.xlabel("Date"); plt.ylabel("Court bookings")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Histogram
plt.figure(figsize=(8, 4))
plt.hist(series, bins=20, color="seagreen", edgecolor="white")
plt.title("Distribution of Daily Court Bookings")
plt.xlabel("Court bookings"); plt.ylabel("Frequency")
plt.show()

In [ ]:
# Rolling mean and standard deviation (7-day window = one full weekly cycle)
roll_mean = series.rolling(window=7).mean()
roll_std = series.rolling(window=7).std()

plt.figure(figsize=(12, 4))
plt.plot(series, label="Original", alpha=0.5)
plt.plot(roll_mean, label="Rolling mean (7d)", color="red")
plt.plot(roll_std, label="Rolling std (7d)", color="black")
plt.title("Rolling Mean & Standard Deviation")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 2. Grid search over alpha, beta, and gamma
The last 20 days are held out as a validation set so every combination is scored on data it has not seen. Seasonal period = 7 (weekly cycle).

In [ ]:
H = 20                       # forecast horizon (days)
SEASON = 7                   # weekly seasonality
train, valid = series[:-H], series[-H:]

grid = np.round(np.arange(0.1, 1.0, 0.1), 1)   # 0.1 ... 0.9
results = []

for a, b, g in itertools.product(grid, grid, grid):
    try:
        m = ExponentialSmoothing(
            train, trend="add", seasonal="add",
            seasonal_periods=SEASON, initialization_method="heuristic"
        ).fit(smoothing_level=a, smoothing_trend=b,
              smoothing_seasonal=g, optimized=False)
        pred = m.forecast(H)
        rmse = np.sqrt(np.mean((valid.values - pred.values) ** 2))
        results.append({"alpha": a, "beta": b, "gamma": g, "RMSE": rmse})
    except Exception:
        continue

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)

## 3. Top 10 best parameter combinations (lowest RMSE on top)

In [ ]:
results_df.head(10)

## 4. Best and chosen parameters

In [ ]:
best = results_df.iloc[0]
alpha_best, beta_best, gamma_best = best["alpha"], best["beta"], best["gamma"]

print("Best / chosen parameters")
print("------------------------")
print(f"alpha (level)    : {alpha_best}")
print(f"beta  (trend)    : {beta_best}")
print(f"gamma (seasonal) : {gamma_best}")
print(f"Validation RMSE  : {best['RMSE']:.4f}")

## 5. Fit the final Holt-Winters model on the full data

In [ ]:
final_model = ExponentialSmoothing(
    series, trend="add", seasonal="add",
    seasonal_periods=SEASON, initialization_method="heuristic"
).fit(smoothing_level=alpha_best, smoothing_trend=beta_best,
      smoothing_seasonal=gamma_best, optimized=False)

fitted = final_model.fittedvalues
print("In-sample RMSE:", np.sqrt(np.mean((series - fitted) ** 2)).round(4))

## 6. Forecast the next 20 days

In [ ]:
forecast = final_model.forecast(H)

plt.figure(figsize=(13, 5))
plt.plot(series, label="Actual", color="steelblue")
plt.plot(fitted, label="Model fitted", color="orange", linestyle="--")
plt.plot(forecast, label="Forecast (next 20 days)", color="red", marker="o", markersize=4)
plt.title("Holt-Winters Seasonal Model - 20-Day Court Booking Forecast")
plt.xlabel("Date"); plt.ylabel("Court bookings")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

forecast.round(1)